In [1]:
# imports

import pandas as pd
import numpy as np
import random
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv("gym_injury_data.csv")
print(df.head())

   Age  Gender       Workout_Type  Workouts_Per_Week  Workout_Duration  \
0   42    Male              Mixed                  1                41   
1   21    Male  Strength Training                  6                85   
2   26    Male             Cardio                  4                80   
3   18  Female               HIIT                  7                31   
4   39    Male              Mixed                  4                30   

  Intensity_Level  Sleep_Hours  Recovery_Days Warm_Up_Completed  \
0             Low          8.5              0               Yes   
1          Medium          4.9              3               Yes   
2          Medium          9.8              4                No   
3          Medium          9.4              1                No   
4             Low          7.8              4               Yes   

  Previous_Injury  Pain_Level_After_Workout Hydration_Level Injury_Risk  
0             Yes                        10            High        High  
1   

In [5]:
# Starting to clean the data. Dropping duplicates and null values.
df = df.drop_duplicates()
df = df.dropna()

In [7]:
# Filtering out unrealistic values based on domain knowledge. For example, ages below 18 or above 60, workouts per week above 7, workout duration above 120 minutes, sleep hours above 12, recovery days above 4, and pain levels above 10 are likely to be errors or outliers.
df = df[
    (df["Age"].between(18, 60)) &
    (df["Workouts_Per_Week"].between(1, 7)) &
    (df["Workout_Duration"].between(20, 120)) &
    (df["Sleep_Hours"].between(0, 12)) &
    (df["Recovery_Days"].between(0, 4)) &
    (df["Pain_Level_After_Workout"].between(0, 10))
]
print("\nData cleaned successfully.")
print("Shape:", df.shape)


Data cleaned successfully.
Shape: (600, 13)


In [11]:
df_encoded = df.copy()

# Mapping Yes/No columns to 1/0 for easier processing by machine learning algorithms.
df_encoded["Warm_Up_Completed"] = df_encoded["Warm_Up_Completed"].map({
    "Yes": 1,
    "No": 0
})

df_encoded["Previous_Injury"] = df_encoded["Previous_Injury"].map({
    "Yes": 1,
    "No": 0
})

# Low / Medium / High columns. Mapping them to 0, 1, 2 for simplicity.
df_encoded["Intensity_Level"] = df_encoded["Intensity_Level"].map({
    "Low": 0,
    "Medium": 1,
    "High": 2
})

df_encoded["Hydration_Level"] = df_encoded["Hydration_Level"].map({
    "Low": 0,
    "Medium": 1,
    "High": 2
})

df_encoded["Injury_Risk"] = df_encoded["Injury_Risk"].map({
    "Low": 0,
    "Medium": 1,
    "High": 2
})
# Gender and Workout_Type columns. Using one-hot encoding to convert them into binary columns.
df_encoded = pd.get_dummies(
    df_encoded,
    columns=["Gender", "Workout_Type"],
    drop_first=True
)

print("\nData encoded successfully.")
print(df_encoded.head())


Data encoded successfully.
   Age  Workouts_Per_Week  Workout_Duration  Intensity_Level  Sleep_Hours  \
0   42                  1                41                0          8.5   
1   21                  6                85                1          4.9   
2   26                  4                80                1          9.8   
3   18                  7                31                1          9.4   
4   39                  4                30                0          7.8   

   Recovery_Days  Warm_Up_Completed  Previous_Injury  \
0              0                  1                1   
1              3                  1                0   
2              4                  0                1   
3              1                  0                0   
4              4                  1                0   

   Pain_Level_After_Workout  Hydration_Level  Injury_Risk  Gender_Male  \
0                        10                2            2         True   
1                       

In [13]:
# Splitting the data into features (X) and target variable (y), and then into training and testing sets.
X = df_encoded.drop("Injury_Risk", axis=1)
y = df_encoded["Injury_Risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=3,
    stratify=y
)

print("\nData split successfully.")
print("Training Rows:", X_train.shape[0])
print("Testing Rows:", X_test.shape[0])


Data split successfully.
Training Rows: 480
Testing Rows: 120


In [15]:
# Defining the machine learning models and their hyperparameters for grid search. Using pipelines to ensure that any necessary preprocessing steps (like scaling) are applied consistently during training and testing.
models = {
    "Decision Tree": {
        "pipeline": Pipeline([
            ("model", DecisionTreeClassifier(random_state=3))
        ]),
        "params": {
            "model__max_depth": [3, 5, 7, None],
            "model__min_samples_leaf": [1, 3, 5]
        }
    },

    "Random Forest": {
        "pipeline": Pipeline([
            ("model", RandomForestClassifier(random_state=3))
        ]),
        "params": {
            "model__n_estimators": [50, 100],
            "model__max_depth": [3, 5, 7, None],
            "model__min_samples_leaf": [1, 3, 5]
        }
    },

    "KNN": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        "params": {
            "model__n_neighbors": [3, 5, 7, 9],
            "model__weights": ["uniform", "distance"]
        }
    },

    "Logistic Regression": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000))
        ]),
        "params": {
            "model__C": [0.01, 0.1, 1, 10]
        }
    }
}

In [16]:
# Running grid search for each model and evaluating their performance on the test set. Keeping track of the best model based on test accuracy.
results = []

best_model = None
best_model_name = ""
best_score = 0

for model_name, model_info in models.items():

    print("\nTesting:", model_name)

    grid = GridSearchCV(
        model_info["pipeline"],
        model_info["params"],
        cv=5,
        scoring="accuracy"
    )

    grid.fit(X_train, y_train)

    y_pred = grid.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    results.append({
        "Model": model_name,
        "Best Parameters": grid.best_params_,
        "Cross Validation Score": grid.best_score_,
        "Test Accuracy": accuracy
    })

    print("Best Parameters:", grid.best_params_)
    print("Accuracy:", accuracy)

    if accuracy > best_score:
        best_score = accuracy
        best_model = grid.best_estimator_
        best_model_name = model_name


Testing: Decision Tree
Best Parameters: {'model__max_depth': 5, 'model__min_samples_leaf': 5}
Accuracy: 0.7333333333333333

Testing: Random Forest
Best Parameters: {'model__max_depth': None, 'model__min_samples_leaf': 3, 'model__n_estimators': 100}
Accuracy: 0.8166666666666667

Testing: KNN
Best Parameters: {'model__n_neighbors': 9, 'model__weights': 'distance'}
Accuracy: 0.65

Testing: Logistic Regression
Best Parameters: {'model__C': 1}
Accuracy: 0.8166666666666667


In [17]:
# Saving the best model to a file for future use.
results_df = pd.DataFrame(results)

print("\nModel Comparison Results:")
print(results_df)

print("\nBest Model:", best_model_name)
print("Best Accuracy:", best_score)


Model Comparison Results:
                 Model                                    Best Parameters  \
0        Decision Tree  {'model__max_depth': 5, 'model__min_samples_le...   
1        Random Forest  {'model__max_depth': None, 'model__min_samples...   
2                  KNN  {'model__n_neighbors': 9, 'model__weights': 'd...   
3  Logistic Regression                                    {'model__C': 1}   

   Cross Validation Score  Test Accuracy  
0                0.739583       0.733333  
1                0.806250       0.816667  
2                0.666667       0.650000  
3                0.795833       0.816667  

Best Model: Random Forest
Best Accuracy: 0.8166666666666667


In [18]:
# Evaluating the best model in more detail using confusion matrix and classification report to understand its performance across different classes.
final_predictions = best_model.predict(X_test)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_predictions))

print("\nClassification Report:")
print(classification_report(y_test, final_predictions))


Confusion Matrix:
[[15  9  0]
 [ 3 40  6]
 [ 0  4 43]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.62      0.71        24
           1       0.75      0.82      0.78        49
           2       0.88      0.91      0.90        47

    accuracy                           0.82       120
   macro avg       0.82      0.79      0.80       120
weighted avg       0.82      0.82      0.81       120



In [20]:
# Saving the best model.
joblib.dump(best_model, "best_gym_injury_model.pkl")

print("\nBest model saved successfully.")


Best model saved successfully.
